<a href="https://colab.research.google.com/github/RyantRamadhan/BigData26_B_2411532001_Imam-Aryanta-Ramadhan/blob/main/Praktikum04/BD_P04_Imam_Aryanta_Ramadhan_2411532001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Menyiapkan Lingkungan dan Data Contoh

In [ ]:
!pip install q duckdb networkx shapely tinydb "deltalake>1.0"
from google.colab import drive
drive.mount("/content/drive")
DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"

import os
os.makedirs (DIR_SIMPAN, exist_ok=True)

TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"
if not os.path.exists(TRIPS_BERSIH):
    TRIPS_BERSIH = "/content/trips_bersih_sintetis"

assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd
# Pengukur waktu (dari Praktikum 1/k-3, versi ringkas)
catatan = []
def ukur (label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round (detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil
df = ukur ("baca trips_bersih", lambda: pd.read_parquet (TRIPS_BERSIH))
# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len (df):,}", "| kolom:", df.shape[1])
# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len (df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample ["waktu_jemput"] = sample["waktu_jemput"].astype(str)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[baca trips_bersih] 0.3375 s
baris: 1,000,000 | kolom: 15


Bila trips_bersih/ Hilang (Jalur Cadangan)

In [ ]:
# Hanya dipakai bila trips_bersih/ Anda hilang: membuat data tiruan dengan kolom yang sama.
import numpy as np, pandas as pd, os, shutil
def bangkitkan_trips_sintetis (keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice (wilayah, size=len (zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})
    pop = np.clip(rng.zipf(1.6, size=len (zona_ids)).astype (float), 1, 60)
    pop = pop / pop.sum()
    pu = rng.choice (zona_ids, size=n, p=pop)
    do = rng.choice (zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta (rng.integers (0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers (2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5+ dist* 2.8+ rng.gamma (2.0, 1.0, n), 2)
    tip = np.round(rng.gamma (1.5, 1.2, n), 2)
    pay = rng.choice ([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])
    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta (durasi, unit="m"),
        "passenger_count": rng.integers (1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd. Series (pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma (0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet (keluar, partition_cols=["borough_naik"], index=False)
    return keluar

TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

Tabel Biasa vs Key-value

In [ ]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan", "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()
#Model 1 relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur ("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))
# Model 2 key-value: kamus (dict) dengan kunci "perjalanan: <id>", nilainya teks JSON
def bangun_kv():
    global kv
    kv = {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}
    return kv
ukur ("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))
# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range (1, len(data) + 1), 200)
def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]
def cari_kv():
    return [json.loads(kv [f"perjalanan:{i}"]) for i in id_uji]
r1 = ukur("cari 200 perjalanan SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan (id_perjalanan)")
r2 = ukur("cari 200 perjalanan SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])
# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND (AVG(total_bayar), 2)
    FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()
def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"]
        total [w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah [w], round (total [w] / jumlah [w], 2)) for w in jumlah], key=lambda x: (-x[1], x[0]))
h1 = ukur("hitung per wilayah SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 1.2368 s
[key-value: bangun dict] 4.3353 s
jumlah key: 300000
[cari 200 perjalanan SQL tanpa indeks] 6.7289 s
[cari 200 perjalanan SQL dengan indeks] 0.0036 s
[cari 200 perjalanan key-value] 0.0011 s
hasil sama? True
[hitung per wilayah SQL GROUP BY] 0.2106 s
[hitung per wilayah key-value (buka semua nilai)] 1.4874 s
[('Manhattan', 116930, 18.26), ('Queens', 81377, 18.29), ('Brooklyn', 62038, 18.24)]
hasil sama? True


Document: Bentuk Data yang Fleksibel

In [ ]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50:", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip:", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip:", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca:", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang:", len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-10 17:57:00",
  "jemput": {
    "zona": 219,
    "wilayah": "Staten Island"
  },
  "tujuan": {
    "zona": 46
  },
  "biaya": {
    "jarak_mil": 2.0,
    "total": 17.99,
    "metode_bayar": "Kartu kredit",
    "tip": 4.81
  }
}
jemput di Manhattan & total > 50: 9
dokumen yang punya field tip: 13979
dokumen yang TIDAK punya tip: 6021
dokumen yang punya field cuaca: 1
total dokumen sekarang: 20001


Penyimpanan per Kolom: DuckDB pada Parquet

In [13]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")

ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())

print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB hitung per wilayah (langsung dari Parquet)] 0.1501 s
  wilayah_jemput  jumlah  rata_bayar
0      Manhattan  391107       18.26
1         Queens  270929       18.27
2       Brooklyn  205487       18.29
3  Staten Island   67041       18.28
4          Bronx   60912       18.21
5            EWR    4524       18.31
[SQLite muat seluruh baris ke tabel] 5.9167 s
[SQLite hitung per wilayah] 1.5284 s
hasil sama? True
[DuckDB hanya wilayah Staten Island] 0.0230 s
[(67041, 18.28)]
Parquet (15 kolom): 32.2 MB | SQLite (5 kolom): 37.6 MB


Graph: Pertanyaan tentang Keterhubungan

In [14]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))

print("node (zona):", G.number_of_nodes(), " edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL:", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50 # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN: continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

h_g = ukur("2 langkah graph (NetworkX)", dua_langkah_graph)
h_s = ukur("2 langkah SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, " terjangkau (graph):", len(h_g), "| (SQL):", len(h_s), " sama?", h_g == h_s)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 0.2325 s
node (zona): 263  edge (arus): 44120
graph: [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
SQL: [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
zona tujuan terpopuler: [(183, 26941), (104, 26916), (146, 26891), (164, 26867), (194, 26847)]
[2 langkah graph (NetworkX)] 0.0219 s
[2 langkah SQL self-join (DuckDB)] 0.0370 s
zona awal: 183  terjangkau (graph): 91 | (SQL): 91  sama? True
dalam <= 1 langkah: 83 zona
dalam <= 2 langkah: 91 zona
dalam <= 3 langkah: 91 zona
dalam <= 4 langkah: 91 zona


Indeks Spasial

In [15]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
lintang = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70) # longitude
bujur = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)      # latitude
titik = points(lintang, bujur)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (lintang >= x - 0.005) & (lintang <= x + 0.005) & (bujur >= y - 0.005) & (bujur <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 loop Python, 5 kotak] 15.4540 s
[cara 2 NumPy, 1000 kotak] 0.7212 s
[bangun indeks STRtree] 0.1043 s
[cara 3 STRtree, 1000 kotak] 0.2495 s
5 kotak pertama loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


Delta Lake: Versi, Time Travel, dan Schema Enforcement

In [17]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

# Diubah ke direktori lokal /content untuk menghindari error file system Google Drive
PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("baris v0:", len(v0), "| v1:", len(v1), " terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), " jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 2517
versi: 2
baris v0: 100000 | v1: 150000  terbaru: 150000
selisih total_bayar (terbaru - v1): 2517.0  jumlah baris dikoreksi: 2517
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


Small File Problem pada Tabel Delta

In [18]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(),
      "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), " versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) # dry_run hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1  versi: 23 | baris: 160000
waktu baca sebelum compact: 0.5034 s | sesudah: 0.4112 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


Merangkum dan Memilih

In [19]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                          langkah   detik
                                baca trips_bersih  0.3375
                           relasional: muat tabel  1.0866
                           key-value: bangun dict  6.3892
             cari 200 perjalanan SQL tanpa indeks  4.7711
            cari 200 perjalanan SQL dengan indeks  0.0037
                    cari 200 perjalanan key-value  0.0011
                  hitung per wilayah SQL GROUP BY  0.2046
  hitung per wilayah key-value (buka semua nilai)  1.4822
                           relasional: muat tabel  1.2368
                           key-value: bangun dict  4.3353
             cari 200 perjalanan SQL tanpa indeks  6.7289
            cari 200 perjalanan SQL dengan indeks  0.0036
                    cari 200 perjalanan key-value  0.0011
                  hitung per wilayah SQL GROUP BY  0.2106
  hitung per wilayah key-value (buka semua nilai)  1.4874
DuckDB hitung per wilayah (langsung dari Parquet)  0.1501
              

Studi Kasus

In [21]:
import pandas as pd

data = [
    ["1. Detail Perjalanan via ID (Seketika)", "Key-Value (Redis/Dict)", "Pencarian tunggal menggunakan key adalah metode tercepat untuk hitungan milidetik (~0.0009 detik) tanpa pemindaian memori.", "Pencarian parameter selain ID (contoh: berdasarkan total_bayar) akan sangat lambat."],
    ["2. Dashboard Analitik per Wilayah", "Columnar (DuckDB + Parquet)", "Proses kalkulasi matriks atau merangkum banyak baris sangat efektif (belasan kali lebih cepat dibanding SQL baris).", "Canggung dan mahal untuk transaksi mengubah / memperbarui baris tunggal secara intensif."],
    ["3. Rekomendasi 2-3 Langkah", "Graph (Neo4j/NetworkX)", "Menjawab relasi interkoneksi jarak node (edges) jauh lebih ringkas dari menumpuk rumusan komputasi JOIN ganda berulang-ulang di SQL.", "Kurang optimal untuk melakukan agregasi sumari/angka keseluruhan tabel."],
    ["4. Radius 500 Meter dari Koordinat", "Spasial (PostGIS/STRtree)", "Indeks (R-Tree) mencegah sistem melacak semua lintang/bujur dari semua riwayat perjalanan di kota.", "Butuh waktu memori (overhead) pada awal penambahan data untuk merakit struktur pohon/indeks."],
    ["5. Audit (Rollback Data Taksi Keuangan)", "Tabel Delta Lake", "Lapisan Lakehouse memberi penjaminan time travel dan schema enforcement, menjaga perombakan finansial yang atomik (ACID).", "Harus rutin dilakukan prosedur VACUUM & compaction agar serpihan riwayat revisi (small files) tak menghabiskan kapasitas disk server."]
]

kolom = ["Kebutuhan / Fitur", "Cara menyimpan yang dipilih", "Alasan", "Kelemahan yang harus diwaspadai"]

df_studi_kasus = pd.DataFrame(data, columns=kolom)

# Menyimpan dataframe menjadi file spreadsheet
df_studi_kasus.to_csv("studi_kasus_taksi.csv", index=False)
df_studi_kasus.to_excel("studi_kasus_taksi.xlsx", index=False)

print("File studi_kasus_taksi.csv dan studi_kasus_taksi.xlsx berhasil dibuat di direktori!")

File studi_kasus_taksi.csv dan studi_kasus_taksi.xlsx berhasil dibuat di direktori!


Latihan 1

In [26]:
import random, time, json
id_uji_2000 = random.sample(range(1, len(sample) + 1), 2000)

t0 = time.perf_counter()
cur = con.cursor()
_ = [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_2000]
waktu_sql_idx = (time.perf_counter() - t0) * 1_000_000 / 2000

t0 = time.perf_counter()
_ = [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]
waktu_kv = (time.perf_counter() - t0) * 1_000_000 / 2000

print(f"Rata-rata cari SQL dengan indeks: {waktu_sql_idx:.2f} mikrodetik")
print(f"Rata-rata cari Key-value        : {waktu_kv:.2f} mikrodetik")

Rata-rata cari SQL dengan indeks: 45.50 mikrodetik
Rata-rata cari Key-value        : 23.53 mikrodetik


Latihan 2

In [28]:
ZONA_CEK = 236
hasil_doc = len(tabel.search((Q.tujuan.zona == ZONA_CEK) & (Q.biaya.metode_bayar == "Tunai")))

hasil_sql = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'", (ZONA_CEK,)).fetchone()[0]
print(f"Jumlah di Dokumen (TinyDB) : {hasil_doc}")
print(f"Jumlah di Tabel SQL        : {hasil_sql}")

Jumlah di Dokumen (TinyDB) : 3
Jumlah di Tabel SQL        : 28


Latihan 3

In [29]:
waktu_1 = ukur("SUM 1 Kolom", lambda: con_d.sql("SELECT SUM(total_bayar) FROM perjalanan").fetchall())
waktu_3 = ukur("SUM 3 Kolom", lambda: con_d.sql("SELECT SUM(total_bayar), SUM(jarak_mil), SUM(zona_jemput) FROM perjalanan").fetchall())

[SUM 1 Kolom] 0.0815 s
[SUM 3 Kolom] 0.1290 s


Latihan 4

In [31]:
# Graph Out-Degree
top_out = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:5]
print("Graph (Node dengan relasi terluar terbanyak):", top_out)

# Relasional / SQL
sql_out = con_d.sql("""
SELECT asal, COUNT(DISTINCT tujuan) AS destinasi_beda
FROM arus GROUP BY asal ORDER BY destinasi_beda DESC LIMIT 5
""").df()
print("SQL:")
print(sql_out)

Graph (Node dengan relasi terluar terbanyak): [(55, 263), (215, 263), (252, 263), (225, 263), (194, 263)]
SQL:
   asal  destinasi_beda
0   144             263
1   252             263
2   164             263
3   138             263
4   146             263


Latihan 5

In [30]:
def uji_indeks(ukuran_kotak):
    k_uji = [box(x - ukuran_kotak/2, y - ukuran_kotak/2, x + ukuran_kotak/2, y + ukuran_kotak/2) for x, y in zip(cx, cy)]
    h_idx = ukur(f"STRtree (Kotak {ukuran_kotak})", lambda: [len(pohon.query(k, predicate="contains")) for k in k_uji])
    return h_idx

uji_indeks(0.02)
uji_indeks(0.001)

[STRtree (Kotak 0.02)] 0.7128 s
[STRtree (Kotak 0.001)] 0.0120 s


[12,
 1,
 2,
 6,
 5,
 6,
 3,
 3,
 5,
 2,
 2,
 9,
 0,
 0,
 7,
 3,
 1,
 1,
 2,
 5,
 9,
 9,
 3,
 2,
 8,
 6,
 0,
 2,
 3,
 7,
 1,
 1,
 2,
 1,
 3,
 1,
 0,
 7,
 5,
 10,
 6,
 3,
 8,
 2,
 9,
 6,
 9,
 2,
 0,
 7,
 1,
 7,
 3,
 7,
 9,
 1,
 3,
 3,
 0,
 4,
 4,
 4,
 12,
 0,
 5,
 1,
 6,
 4,
 5,
 6,
 2,
 5,
 3,
 1,
 3,
 0,
 2,
 0,
 5,
 2,
 2,
 1,
 0,
 5,
 8,
 1,
 9,
 12,
 4,
 0,
 2,
 1,
 0,
 0,
 8,
 1,
 13,
 9,
 10,
 1,
 0,
 16,
 3,
 1,
 1,
 2,
 11,
 3,
 12,
 7,
 5,
 1,
 11,
 5,
 6,
 5,
 1,
 4,
 2,
 4,
 5,
 7,
 2,
 0,
 11,
 2,
 2,
 9,
 0,
 4,
 6,
 3,
 7,
 13,
 2,
 6,
 2,
 0,
 10,
 3,
 3,
 3,
 6,
 4,
 4,
 7,
 1,
 1,
 4,
 6,
 6,
 1,
 10,
 1,
 7,
 5,
 12,
 10,
 5,
 2,
 3,
 5,
 1,
 8,
 2,
 8,
 5,
 1,
 0,
 7,
 2,
 7,
 12,
 3,
 6,
 1,
 5,
 3,
 2,
 0,
 4,
 0,
 3,
 1,
 1,
 1,
 1,
 9,
 10,
 1,
 4,
 5,
 2,
 2,
 6,
 9,
 10,
 4,
 5,
 4,
 5,
 0,
 6,
 1,
 3,
 3,
 7,
 1,
 4,
 0,
 5,
 4,
 4,
 8,
 2,
 1,
 3,
 12,
 7,
 1,
 2,
 3,
 1,
 4,
 8,
 3,
 1,
 2,
 1,
 0,
 11,
 0,
 2,
 0,
 6,
 2,
 4,
 6,
 4,
 15,
 4,
 9,
 2,
 5,
 5

Latihan 6

In [33]:
dt = DeltaTable (PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore (1)
dt = DeltaTable (PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 24 | baris: 150000
versi setelah restore: 25 | baris: 150000
['RESTORE', 'RESTORE', 'OPTIMIZE']
